# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s for data discovery.

In [ ]:
# List available RecordSet entities by @id
record_set_ids = [rs['@id'] for rs in metadata.to_json().get('recordSet', [])]
print("Available record sets (@id):")
for rsid in record_set_ids:
    print(" -", rsid)

# For each RecordSet, list its fields and columns by @id
for rs in metadata.to_json().get('recordSet', []):
    print(f"\nRecordSet @id: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Fields:")
        for field in fields:
            fid = field['@id'] if isinstance(field, dict) and '@id' in field else field
            print(f"    - {fid}")
    if 'column' in rs:
        columns = rs['column'] if isinstance(rs['column'], list) else [rs['column']]
        print("  Columns:")
        for col in columns:
            cid = col['@id'] if isinstance(col, dict) and '@id' in col else col
            print(f"    - {cid}")

## 3. Data Extraction
Load data from selected record sets into DataFrames for analysis using record set and field `@id`s.

In [ ]:
# Extract data from each record set
# Update the record_set_ids to match what was listed above; else, fallback to single RecordSet
if record_set_ids:
    selected_record_sets = record_set_ids
else:
    # Fallback/default: if no record sets found
    selected_record_sets = []

dataframes = {}

for rsid in selected_record_sets:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded DataFrame for RecordSet {rsid}: shape={df.shape}")

# Pick the first record set for demonstration
if selected_record_sets:
    main_rs = selected_record_sets[0]
    print(f"Columns in record set {main_rs}:")
    print(dataframes[main_rs].columns.tolist())
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing: filtering, normalization, and grouping for analysis. Use field `@id`s from the overview.

In [ ]:
# Pick a numeric field for demonstration (substitute this with an actual @id from your field listing for real use)
if selected_record_sets:
    main_df = dataframes[main_rs]
    # Try to find a numeric column, otherwise use a placeholder
    numeric_candidates = [c for c in main_df.columns if main_df[c].dtype in [int, float] or pd.api.types.is_numeric_dtype(main_df[c])]
    if not numeric_candidates:
        print("No numeric fields found in the primary record set. Please select one by @id from your dataset schema.")
        numeric_field = None
    else:
        numeric_field = numeric_candidates[0]
    print(f"Using numeric field (by @id/column name): {numeric_field}")
    
    threshold = main_df[numeric_field].quantile(0.7) if numeric_field else None
    if numeric_field:
        filtered_df = main_df[main_df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        normalized_col = f"{numeric_field}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() or 1)
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, normalized_col]].head())
    
    # Try grouping by a categorical/text field
    group_candidates = [c for c in main_df.columns if main_df[c].dtype == object and c != numeric_field]
    group_field = group_candidates[0] if group_candidates else None
    if group_field and numeric_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions and field relationships using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if selected_record_sets and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=main_df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:
- Load a dataset using a Croissant schema via `mlcroissant`
- Examine record set and field structure via their `@id`s
- Load, explore, and visualize data for further downstream analysis

**You are encouraged to further extend this notebook with domain-specific analysis to gain deeper insights into extracellular vesicle mass spectrometry data.**